# E9 · Адаптер B0 → внутренний контекст B5

**Теперь действительно обучаем маленькую сеть, но B0 и B5 остаются заморожены.** Адаптер по карте F3 энкодера B0 предсказывает середину и выход stage 3 B5.

Это диагностический эксперимент с подсказками настоящей B5: ранние стадии B5, вход выбранных токенов в stage 3 и остальные ветви декодера остаются настоящими. Он проверяет добавочную потерю качества при замене контекста и восстановления выхода вне регионов. **Это ещё не ускоренный каскад, измерять по этому ноутбуку его скорость нельзя.**

Используем тот же запуск `e8_cityscapes_v1`, кэш B0/B5 и признаки entropy+kNN. Кэш E0 и E8/E8b заново не считаем. Только Cityscapes. ACDC не используется ни для обучения, ни для выбора эпохи.

## Как запустить
1. Открой этот notebook в Colab, включи GPU (T4 достаточно для старта), сохрани копию на Drive.
2. Если предыдущая среда потеряна, выполни **подготовительные ячейки ниже**: они восстанавливают архивы, исходный код и готовый кэш. Если работаешь в той же среде, можешь также выполнить их: готовые файлы сохраняются.
3. Проверь путь `/content/drive/MyDrive/datasets/cityscapes` к `val.zip` и `gtFine_trainvaltest.zip`.
4. Запускай все ячейки по порядку.
5. Новый результат появится в `Мой диск/e8_cityscapes_v1/e9_adapter_v1/`.

Новые teacher-цели занимают дополнительное место на Drive: сохраняются float16-карты для 500 кадров (порядка 1–2 ГБ до/после сжатия зависит от признаков). Они нужны для продолжения после сброса VM. Скрипт не удаляет старые данные.


In [ ]:
# 1. Окружение. PyTorch/CUDA не переустанавливаем.
import os, sys, subprocess
os.environ['USE_TF'] = '0'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'transformers==4.46.3', 'cityscapesscripts==2.2.4', 'scikit-learn', 'tqdm'], check=True)
import torch
assert torch.cuda.is_available(), 'Включи GPU: Среда выполнения → Сменить среду выполнения'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# 2. Укажи папку с архивами. Обычно нужно поменять ТОЛЬКО эту строку.
from pathlib import Path
ARCHIVE_DIR = Path('/content/drive/MyDrive/datasets/cityscapes')  # <-- твоя папка или ярлык
IMAGE_ZIP = ARCHIVE_DIR / 'val.zip'
GT_ZIP = ARCHIVE_DIR / 'gtFine_trainvaltest.zip'

# Если архив называется иначе, поменяй IMAGE_ZIP/GT_ZIP выше.
for p in [IMAGE_ZIP, GT_ZIP]:
    assert p.is_file(), f'Не найден {p}. Найди архив в панели Files → drive → MyDrive и скопируй путь.'

RUN_NAME = 'e8_cityscapes_v1'  # для нового независимого запуска используй другое имя
SAVE = Path('/content/drive/MyDrive') / RUN_NAME
WORK = Path('/content') / RUN_NAME
EXP = WORK / 'experiments'
DATA = WORK / 'data'
CACHE = WORK / 'cache'
RESULTS = SAVE / 'results'
for p in [SAVE, EXP, DATA, CACHE, RESULTS]: p.mkdir(parents=True, exist_ok=True)
os.environ.update(E8_DATA=str(DATA), E8_CACHE=str(CACHE), E8_RESULTS=str(RESULTS), E8_BACKUP=str(SAVE/'backups'))
print('Результаты:', SAVE)


## Распаковка только validation
Поддерживаются `val/<город>/...` и `leftImg8bit/val/<город>/...` в архиве изображений. Из разметки берём только `gtFine/val/*/*_labelIds.png`. Полный train/test не распаковывается.

In [ ]:
# 3. Извлечение и проверка 500 пар.
import zipfile, shutil
from collections import Counter

def unpack_cityscapes(archive, image):
    count = 0
    with zipfile.ZipFile(archive) as z:
        for member in z.infolist():
            parts = Path(member.filename).parts
            suffix = '_leftImg8bit.png' if image else '_gtFine_labelIds.png'
            if member.is_dir() or not member.filename.endswith(suffix) or 'val' not in parts:
                continue
            k = parts.index('val')
            tail = parts[k:]
            if len(tail) != 3 or '..' in tail: continue
            branch = 'leftImg8bit' if image else 'gtFine'
            out = DATA/'cityscapes'/branch/Path(*tail)
            out.parent.mkdir(parents=True, exist_ok=True)
            if not out.exists() or out.stat().st_size != member.file_size:
                tmp = out.with_suffix(out.suffix+'.part')
                with z.open(member) as src, tmp.open('wb') as dst: shutil.copyfileobj(src,dst)
                tmp.replace(out)
            count += 1
    return count

print('Изображений:', unpack_cityscapes(IMAGE_ZIP, True))
print('Масок:', unpack_cityscapes(GT_ZIP, False))
images = sorted((DATA/'cityscapes/leftImg8bit/val').glob('*/*_leftImg8bit.png'))
counts = Counter(p.parent.name for p in images)
assert dict(counts) == {'frankfurt':267, 'lindau':59, 'munster':174}, counts
for p in images:
    gt = DATA/'cityscapes/gtFine/val'/p.parent.name/p.name.replace('_leftImg8bit.png','_gtFine_labelIds.png')
    assert gt.is_file(), f'Отсутствует маска {gt}'
print('500 пар OK:', dict(counts))


In [ ]:
# 4. Создание автономных скриптов (никакие файлы репозитория не меняются).
import json, hashlib
SOURCES = {'common.py': 'import os\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nfrom PIL import Image\nfrom torch.utils.data import Dataset\nfrom transformers import SegformerForSemanticSegmentation\nfrom cityscapesscripts.helpers.labels import labels\n\nROOT = Path(__file__).resolve().parent\nDATA = Path(os.environ["E8_DATA"])\nCACHE = Path(os.environ["E8_CACHE"])\nRESULTS = Path(os.environ["E8_RESULTS"])\n\nSIZE = (512, 1024)\nNUM_CLASSES = 19\nIGNORE = 255\n\nMODELS = {\n    "b0": "nvidia/segformer-b0-finetuned-cityscapes-512-1024",\n    "b5": "nvidia/segformer-b5-finetuned-cityscapes-1024-1024",\n}\n\n# frankfurt is used to fit scores/routers, the rest is test only\nDOMAINS = ["cs_fit", "cs_test", "fog", "night", "rain", "snow"]\n\n_LUT = np.full(256, IGNORE, np.uint8)\nfor l in labels:\n    if 0 <= l.id < 256 and l.trainId not in (-1, 255):\n        _LUT[l.id] = l.trainId\n\n\ndef list_domain(domain):\n    """Returns sorted (image, gt) path pairs for a domain."""\n    if domain.startswith("cs_"):\n        cities = ["frankfurt"] if domain == "cs_fit" else ["lindau", "munster"]\n        imgs = [p for c in cities for p in (DATA / "cityscapes/leftImg8bit/val" / c).glob("*.png")]\n        gt = lambda p: Path(str(p).replace("leftImg8bit", "gtFine", 1)\n                            .replace("_leftImg8bit.png", "_gtFine_labelIds.png"))\n    else:\n        imgs = list((DATA / "acdc/rgb_anon" / domain / "val").rglob("*_rgb_anon.png"))\n        gt = lambda p: Path(str(p).replace("rgb_anon", "gt", 1)\n                            .replace("_rgb_anon.png", "_gt_labelTrainIds.png"))\n    return [(p, gt(p)) for p in sorted(imgs)]\n\n\ndef image_tensor(image):\n    # Tensor bilinear antialiased resize, matching the original torchvision v2 intent.\n    import torch.nn.functional as F\n    a = np.array(image, dtype=np.uint8, copy=True)\n    x = torch.from_numpy(a).permute(2,0,1).float().div(255)\n    x = F.interpolate(x[None], size=SIZE, mode="bilinear", align_corners=False, antialias=True)[0]\n    mean = torch.tensor([.485,.456,.406])[:,None,None]\n    std = torch.tensor([.229,.224,.225])[:,None,None]\n    return (x-mean)/std\n\n\nclass SegSet(Dataset):\n    def __init__(self, domain):\n        self.domain = domain\n        self.items = list_domain(domain)\n        self.img_tf = image_tensor\n\n\n    def __len__(self):\n        return len(self.items)\n\n    def __getitem__(self, i):\n        img_path, gt_path = self.items[i]\n        gt = np.array(Image.open(gt_path))\n        if self.domain.startswith("cs_"):\n            gt = _LUT[gt]\n        gt = Image.fromarray(gt).resize(SIZE[::-1], Image.NEAREST)\n        return self.img_tf(Image.open(img_path).convert("RGB")), torch.from_numpy(np.array(gt)), img_path.stem\n\n\ndef load_model(name, device="cuda"):\n    model = SegformerForSemanticSegmentation.from_pretrained(MODELS[name])\n    return model.half().to(device).eval()\n\n\ndef confusion(pred, gt):\n    m = gt != IGNORE\n    return np.bincount(NUM_CLASSES * gt[m].astype(np.int64) + pred[m],\n                       minlength=NUM_CLASSES ** 2).reshape(NUM_CLASSES, NUM_CLASSES)\n\n\ndef miou(conf):\n    iou = np.diag(conf) / (conf.sum(0) + conf.sum(1) - np.diag(conf) + 1e-9)\n    return 100 * iou[conf.sum(1) > 0].mean()\n\n\nTILE = 64  # 512x1024 -> 8x16 tiles\n\n\ndef load(domain, *keys):\n    """Yields cached arrays of every image in a domain, in a fixed order."""\n    for p in sorted((CACHE / domain).glob("*.npz")):\n        with np.load(p) as z:\n            yield [z[k] for k in keys]\n\n\ndef tile_sum(x):\n    h, w = x.shape\n    return x.reshape(h // TILE, TILE, w // TILE, TILE).sum((1, 3))\n\n\ndef gain(gt, pred_b0, pred_b5):\n    """+1 where only B5 is right, -1 where only B0 is right, 0 elsewhere or on ignore."""\n    return ((pred_b5 == gt).astype(np.int8) - (pred_b0 == gt)) * (gt != IGNORE)\n\n\ndef top_tiles(scores, budget):\n    """Boolean mask of the top `budget` share of tiles by score, over the whole domain."""\n    order = np.argsort(-scores.ravel(), kind="stable")\n    m = np.zeros(scores.size, bool)\n    m[order[:round(budget * scores.size)]] = True\n    return m.reshape(scores.shape)\n\n\ndef route_confs(domain, scores, budgets):\n    """Per-image confusions (budgets x images x C x C) when top_tiles get B5."""\n    masks = [top_tiles(scores, b) for b in budgets]\n    confs = np.zeros((len(budgets), len(scores), NUM_CLASSES, NUM_CLASSES), np.int64)\n    for i, (gt, b0, b5) in enumerate(load(domain, "gt", "pred_b0", "pred_b5")):\n        for j, m in enumerate(masks):\n            px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n            confs[j, i] = confusion(np.where(px, b5, b0), gt)\n    return confs\n\n\ndef route_curve(domain, scores, budgets):\n    """mIoU per budget, see route_confs."""\n    return [miou(c.sum(0)) for c in route_confs(domain, scores, budgets)]\n', 'e2_confidence.py': '"""E2: standard confidence scores of B0 as routing signals.\n\nPixel AUROC (logit resolution, sampled pixels) for two targets:\n  err  - B0 is wrong\n  help - B0 is wrong and B5 is right\nTile routing: mean score per tile, % of the B5-B0 gap closed (as in E1).\n"""\nimport json\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom sklearn.metrics import roc_auc_score\n\nfrom common import DOMAINS, IGNORE, RESULTS, TILE, load, route_curve\n\nBUDGETS = [0.05, 0.1, 0.2, 0.3]\nPIXELS_PER_IMAGE = 2000\nSTRIDE = 4  # logits are at 1/4 resolution\n\n\ndef uncertainty(logits):\n    """Per-pixel scores, higher = less confident."""\n    p = logits.softmax(0)\n    top2 = p.topk(2, dim=0).values\n    return {\n        "entropy": -(p * p.clamp_min(1e-12).log()).sum(0),\n        "msp": 1 - top2[0],\n        "margin": top2[1] - top2[0],\n        "energy": -logits.logsumexp(0),\n    }\n\n\ndef main():\n    base = json.loads((RESULTS / "e0.json").read_text())\n    oracle = json.loads((RESULTS / "e1.json").read_text())\n    rng = np.random.default_rng(0)\n    auroc, closed = {}, {}\n\n    for domain in DOMAINS:\n        tiles, pix, err, helps = {}, {}, [], []\n        for gt, b0, b5, logits in load(domain, "gt", "pred_b0", "pred_b5", "logits_b0"):\n            scores = uncertainty(torch.from_numpy(logits).float())\n            for k, s in scores.items():\n                tiles.setdefault(k, []).append(F.avg_pool2d(s[None], TILE // STRIDE)[0].numpy())\n\n            gt, b0, b5 = (a[STRIDE // 2::STRIDE, STRIDE // 2::STRIDE] for a in (gt, b0, b5))\n            valid = np.flatnonzero(gt != IGNORE)\n            idx = rng.choice(valid, min(PIXELS_PER_IMAGE, valid.size), replace=False)\n            for k, s in scores.items():\n                pix.setdefault(k, []).append(s.numpy().ravel()[idx])\n            wrong = b0.ravel()[idx] != gt.ravel()[idx]\n            err.append(wrong)\n            helps.append(wrong & (b5.ravel()[idx] == gt.ravel()[idx]))\n\n        err, helps = np.concatenate(err), np.concatenate(helps)\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        auroc[domain], closed[domain] = {}, {}\n        for k in tiles:\n            s = np.concatenate(pix[k])\n            auroc[domain][k] = [round(roc_auc_score(err, s), 3), round(roc_auc_score(helps, s), 3)]\n            curve = route_curve(domain, np.stack(tiles[k]), BUDGETS)\n            closed[domain][k] = [round(100 * (m - b0_miou) / (b5_miou - b0_miou), 1) for m in curve]\n\n    print(f"{\'domain\':8} {\'method\':8} {\'AUC err\':>8} {\'AUC help\':>9}" + "".join(f"{b:>7.0%}" for b in BUDGETS))\n    for domain in DOMAINS:\n        for k, (a_err, a_help) in auroc[domain].items():\n            print(f"{domain:8} {k:8} {a_err:8.3f} {a_help:9.3f}" + "".join(f"{c:7.1f}" for c in closed[domain][k]))\n        print(f"{domain:8} {\'oracle\':8} {\'\':8} {\'\':9}" + "".join(f"{c:7.1f}" for c in oracle[domain]["oracle_tile"][:len(BUDGETS)]))\n\n    (RESULTS / "e2.json").write_text(json.dumps({"budgets": BUDGETS, "auroc": auroc, "closed": closed}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n', 'e8_cityscapes.py': '"""E8: gate for the adapter idea - is a frozen stage-3 context good enough for B5?\n\nB5 sees the whole frame, but in the chosen stages only the tokens of routed tiles are updated\nby the transformer blocks. Every other token keeps its stage-entry value for all blocks, so it\nonly serves as keys/values ("frozen context"). Here the entry values come from the real B5\n(an upper bound for any adapter that maps B0 tokens into B5 token space).\n\nQuality only: the whole frame is still computed, only the write-back is masked. The saving is\ncounted analytically for now (blocks run on routed queries only; context costs K/V projections),\nreal FLOPs and latency are measured later.\n\nStages (0-based): 2 = stride 16, 40 blocks, most of B5\'s encoder compute; 3 = stride 32.\nVariants: "s3" freezes stage 2 only, "s3+s4" freezes stages 2 and 3. Stages 0-1 are run on the\nwhole frame in this experiment (known simplification, about a fifth of the encoder cost).\n\nTiles are selected as in E5 (domain-wide ranking at 10% and 20%): `rank` (entropy + knn) and the oracle.\nReported as % of the B5-B0 gap closed, next to:\n  full  - B5 on the full frame inside the routed tiles (the E5 "full" column, the target)\n  agree - pixel agreement with that full-frame B5 prediction inside the routed tiles\nGate: s3 should stay close to `full` on ACDC. For reference E5 ctx64 (cost x0.9) closed 42 / 17 / 35 / 60\n(fog / night / rain / snow) against 61 / 30 / 59 / 59 for full.\n"""\nimport json\nfrom types import SimpleNamespace\n\nimport numpy as np\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nfrom common import (CACHE, DOMAINS, NUM_CLASSES, RESULTS, SIZE, TILE, SegSet, confusion, gain, load,\n                    load_model, miou, tile_sum, top_tiles)\ndef rank(scores):\n    return scores.ravel().argsort().argsort().reshape(scores.shape)\n\nDOMAINS = ["cs_fit", "cs_test"]\n\nBUDGETS = [0.1, 0.2]\nSTAGES = {"s3": [2], "s3+s4": [2, 3]}\n\n\nclass Frozen(nn.Module):\n    """Wraps a SegformerLayer: tokens outside state.mask are reset to the stage entry value."""\n\n    def __init__(self, layer, state, first):\n        super().__init__()\n        self.layer, self.state, self.first = layer, state, first\n\n    def forward(self, hidden_states, *args, **kwargs):\n        s = self.state\n        if self.first:\n            s.entry = hidden_states\n        out = self.layer(hidden_states, *args, **kwargs)\n        if s.mask is None:\n            return out\n        return (torch.where(s.mask, out[0], s.entry),) + tuple(out[1:])\n\n\ndef install(model, stages=(2, 3)):\n    """Wrap every block of the given stages, returns {stage: state}."""\n    state = {st: SimpleNamespace(mask=None, entry=None) for st in stages}\n    blocks = model.segformer.encoder.block\n    for st in stages:\n        for i, layer in enumerate(blocks[st]):\n            blocks[st][i] = Frozen(layer, state[st], i == 0)\n    return state\n\n\ndef token_mask(m, stage):\n    """(8, 16) bool tile mask -> (1, N, 1) bool mask over the tokens of a stage (stride 4 * 2**stage)."""\n    k = TILE // (4 * 2 ** stage)\n    t = torch.from_numpy(m).cuda().repeat_interleave(k, 0).repeat_interleave(k, 1)\n    return t.reshape(1, -1, 1)\n\n\n@torch.inference_mode()\ndef predict(model, state, img, masks):\n    """B5 prediction on the full frame where only `masks` {stage: token mask} tokens are updated."""\n    for st, s in state.items():\n        s.mask = masks.get(st)\n    try:\n        logits = model(pixel_values=img[None].cuda().half()).logits\n        return F.interpolate(logits.float(), SIZE, mode="bilinear", align_corners=False).argmax(1)[0].byte().cpu().numpy()\n    finally:\n        for s in state.values():\n            s.mask = None\n            s.entry = None\n\n\n\ndef sanity(model, state):\n    """An all-true mask must reproduce the cached full-frame B5 prediction."""\n    img = SegSet("cs_fit")[0][0]\n    (cached,) = next(load("cs_fit", "pred_b5"))\n    full = {st: token_mask(np.ones((SIZE[0] // TILE, SIZE[1] // TILE), bool), st) for st in state}\n    agree = (predict(model, state, img, full) == cached).mean()\n    assert agree > 0.999, f"all-true mask does not reproduce B5, agreement {agree:.4f}"\n\n\ndef main():\n    model = load_model("b5")\n    state = install(model)\n    sanity(model, state)\n\n    base = json.loads((RESULTS / "e0_cityscapes.json").read_text())\n    feats = np.load(CACHE / "e8_tiles_cityscapes.npz")\n    report_path = RESULTS / "e8_cityscapes.json"\n    report = json.loads(report_path.read_text()) if report_path.exists() else {}\n\n    print(f"{\'domain\':8} {\'method\':8} {\'budget\':>6} {\'full\':>7}"\n          + "".join(f"{n:>9}" for n in STAGES) + "".join(f"{f\'agree {n}\':>14}" for n in STAGES))\n    for domain in DOMAINS:\n        if domain in report:\n            print(domain, "уже завершён, пропускаем", flush=True)\n            continue\n        dataset = SegSet(domain)\n        assert [p.stem for p, _ in dataset.items] == [p.stem for p in sorted((CACHE / domain).glob("*.npz"))]\n\n        entropy, knn = feats[f"{domain}__entropy__mean"], feats[f"{domain}__knn__mean"]\n        gains = np.stack([tile_sum(gain(*a)) for a in load(domain, "gt", "pred_b0", "pred_b5")])\n        scores = {"rank": rank(entropy) + rank(knn), "oracle": gains}\n        masks = {(k, b): top_tiles(s, b) for k, s in scores.items() for b in BUDGETS}\n\n        zeros = lambda: np.zeros((NUM_CLASSES, NUM_CLASSES), np.int64)\n        full = {key: zeros() for key in masks}\n        conf = {(*key, n): zeros() for key in masks for n in STAGES}\n        agree = {(*key, n): np.zeros(2, np.int64) for key in masks for n in STAGES}\n\n        started = __import__("time").monotonic()\n        loader = DataLoader(dataset, batch_size=None, num_workers=2)\n        for i, ((img, _, _), (gt, b0, b5)) in enumerate(zip(loader, load(domain, "gt", "pred_b0", "pred_b5"))):\n            for key, m in masks.items():\n                px = m[i].repeat(TILE, 0).repeat(TILE, 1)\n                full[key] += confusion(np.where(px, b5, b0), gt)\n                for n, stages in STAGES.items():\n                    pred = predict(model, state, img, {st: token_mask(m[i], st) for st in stages})\n                    conf[(*key, n)] += confusion(np.where(px, pred, b0), gt)\n                    agree[(*key, n)] += [(pred == b5)[px].sum(), px.sum()]\n\n            if (i+1) % 5 == 0 or i+1 == len(dataset):\n                elapsed = __import__("time").monotonic()-started\n                eta = elapsed/(i+1)*(len(dataset)-i-1)/60\n                print(f"E8 {domain}: {i+1}/{len(dataset)}, {elapsed/60:.1f} мин, ETA ~{eta:.1f} мин", flush=True)\n\n        b0_miou, b5_miou = base[domain]["b0"], base[domain]["b5"]\n        closed = lambda c: round(100 * (miou(c) - b0_miou) / (b5_miou - b0_miou), 1)\n        report[domain] = {}\n        for key in masks:\n            row = {"full": closed(full[key])}\n            row.update({n: closed(conf[(*key, n)]) for n in STAGES})\n            row.update({f"agree_{n}": round(100 * agree[(*key, n)][0] / agree[(*key, n)][1], 1) for n in STAGES})\n            report[domain][f"{key[0]}@{key[1]}"] = row\n            print(f"{domain:8} {key[0]:8} {key[1]:6.0%} {row[\'full\']:7.1f}"\n                  + "".join(f"{row[n]:9.1f}" for n in STAGES)\n                  + "".join(f"{row[f\'agree_{n}\']:14.1f}" for n in STAGES))\n\n        temp = report_path.with_suffix(".json.part")\n        temp.write_text(json.dumps(report, indent=2))\n        temp.replace(report_path)\n        print("Сохранён домен:", domain, flush=True)\n\n    print("Результат:", report_path, flush=True)\n\n\nif __name__ == "__main__":\n    main()\n', 'runner.py': '"""Cityscapes-only preparation and E8; frozen B0/B5. No ACDC tuning."""\nimport os\nos.environ[\'USE_TF\'] = \'0\'\nimport argparse, gc, hashlib, json, platform, tarfile, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom common import CACHE, RESULTS, SIZE, TILE, SegSet, load, load_model, confusion, miou\n\nDOMAINS = [\'cs_fit\', \'cs_test\']\nSIGNATURE = \'e8-cs-colab-v1\'\n\ndef write_json(path, data):\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_suffix(path.suffix + \'.part\')\n    tmp.write_text(json.dumps(data, indent=2))\n    tmp.replace(path)\n\ndef checkpoint(domain):\n    backup = os.environ.get(\'E8_BACKUP\')\n    if not backup:\n        return\n    dest = Path(backup); dest.mkdir(parents=True, exist_ok=True)\n    name = dest / f\'cache_{domain}.tar\'\n    tmp = name.with_suffix(\'.tar.part\')\n    with tarfile.open(tmp, \'w\') as tar:\n        tar.add(CACHE / domain, arcname=domain)\n    tmp.replace(name)\n    print(\'Кэш сохранён на Drive:\', name, flush=True)\n\ndef valid_cache(path):\n    if not path.exists():\n        return False\n    with np.load(path, allow_pickle=False) as z:\n        assert set([\'gt\', \'pred_b0\', \'pred_b5\', \'logits_b0\', \'feat_b0\']) <= set(z.files)\n        assert all(z[k].shape == SIZE for k in [\'gt\', \'pred_b0\', \'pred_b5\'])\n        assert z[\'logits_b0\'].shape == (19, 128, 256)\n        assert z[\'feat_b0\'].shape == (256, 64, 128)\n    return True\n\n@torch.inference_mode()\ndef cache():\n    models = [load_model(n) for n in [\'b0\', \'b5\']]\n    captured = {}\n    handle = models[0].decode_head.classifier.register_forward_pre_hook(lambda m, x: captured.update(f=x[0]))\n    report = {}\n    try:\n        for domain in DOMAINS:\n            data = SegSet(domain)\n            assert len(data) == (267 if domain == \'cs_fit\' else 233)\n            folder = CACHE / domain; folder.mkdir(parents=True, exist_ok=True)\n            confs = [np.zeros((19,19), np.int64) for _ in range(2)]\n            start = time.monotonic()\n            for i in range(len(data)):\n                img, gt, name = data[i]\n                path = folder / f\'{name}.npz\'\n                if not valid_cache(path):\n                    x = img[None].cuda().half()\n                    l0 = models[0](pixel_values=x).logits\n                    feat = F.avg_pool2d(captured[\'f\'], 2)[0].cpu().numpy()\n                    preds = []\n                    for logits in [l0, models[1](pixel_values=x).logits]:\n                        preds.append(F.interpolate(logits.float(), SIZE, mode=\'bilinear\', align_corners=False).argmax(1)[0].byte().cpu().numpy())\n                    tmp = path.with_suffix(\'.npz.part\')\n                    with tmp.open(\'wb\') as f:\n                        np.savez_compressed(f, gt=gt.numpy(), pred_b0=preds[0], pred_b5=preds[1], logits_b0=l0[0].cpu().numpy(), feat_b0=feat)\n                    tmp.replace(path)\n                with np.load(path) as z:\n                    for j, k in enumerate([\'pred_b0\', \'pred_b5\']):\n                        confs[j] += confusion(z[k], z[\'gt\'])\n                if (i+1) % 10 == 0 or i+1 == len(data):\n                    print(f\'E0 {domain}: {i+1}/{len(data)}, {(time.monotonic()-start)/60:.1f} мин\', flush=True)\n            report[domain] = dict(zip([\'b0\',\'b5\'], [float(miou(c)) for c in confs]))\n            write_json(RESULTS / \'e0_cityscapes.json\', report)\n            checkpoint(domain)\n    finally:\n        handle.remove()\n    print(report, flush=True)\n\n@torch.inference_mode()\ndef scores():\n    """Only entropy + kNN required by E8. Bank fitted solely on cs_fit."""\n    from e2_confidence import uncertainty\n    from common import IGNORE\n    reference = CACHE / \'e8_knn_bank.npy\'\n    if not reference.exists():\n        rng = np.random.default_rng(0)\n        samples = []\n        for gt, feat in load(\'cs_fit\', \'gt\', \'feat_b0\'):\n            y = gt[4::8,4::8].ravel()\n            valid = np.flatnonzero(y != IGNORE)\n            idx = rng.choice(valid, min(500, len(valid)), replace=False)\n            samples.append(feat.reshape(256,-1).T[idx].astype(np.float32))\n        x = np.concatenate(samples)\n        # Separate seeded RNG: exact bank saved, not claimed identical to old E3.\n        idx = np.random.default_rng(0).permutation(len(x))[:20000]\n        bank = F.normalize(torch.from_numpy(x[idx]), dim=1).numpy()\n        np.save(reference, bank)\n    bank = torch.from_numpy(np.load(reference)).cuda().float()\n    assert bank.shape == (20000,256)\n    output = {}\n    for domain in DOMAINS:\n        arrays = {\'entropy\':[], \'knn\':[]}\n        count = len(SegSet(domain))\n        for i, (logits, feat) in enumerate(load(domain, \'logits_b0\',\'feat_b0\')):\n            entropy = uncertainty(torch.from_numpy(logits).float())[\'entropy\']\n            entropy = F.avg_pool2d(entropy[None],2)[0].numpy()\n            f = F.normalize(torch.from_numpy(feat.astype(np.float32)).cuda().flatten(1).T, dim=1)\n            distances = []\n            for chunk in f.split(512):\n                distances.append((1-(chunk @ bank.T).topk(10,dim=1).values.mean(1)).cpu())\n            knn = torch.cat(distances).numpy().reshape(64,128)\n            for name, m in [(\'entropy\',entropy),(\'knn\',knn)]:\n                # 8x8 feature cells per 64x64 tile.\n                arrays[name].append(m.reshape(8,8,16,8).mean((1,3)))\n            if (i+1)%10==0 or i+1==count:\n                print(f\'kNN {domain}: {i+1}/{count}\', flush=True)\n        for k,v in arrays.items():\n            output[f\'{domain}__{k}__mean\'] = np.stack(v)\n    path = CACHE / \'e8_tiles_cityscapes.npz\'\n    np.savez(path, **output)\n    write_json(RESULTS/\'knn_config.json\', {\'fit\':\'cs_fit only\', \'bank\':20000, \'k\':10, \'sampling_seed\':0,\n        \'bank_sha256\':hashlib.sha256(reference.read_bytes()).hexdigest(),\n        \'note\':\'Minimal E3 alternative; saved bank differs from historical E3 RNG. Do not compare as exact reproduction.\'})\n    print(\'Сохранено:\', path, flush=True)\n\n@torch.inference_mode()\ndef smoke():\n    from e8_cityscapes import install, predict, token_mask\n    model = load_model(\'b5\'); state = install(model)\n    data = SegSet(\'cs_fit\'); img, _, name = data[0]\n    all_masks = {st: token_mask(np.ones((8,16),bool),st) for st in state}\n    plain = predict(model,state,img,{})\n    masked = predict(model,state,img,all_masks)\n    assert np.array_equal(plain,masked), \'All-true differs from unmasked model\'\n    path = CACHE/\'cs_fit\'/f\'{name}.npz\'\n    assert path.is_file(), \'Сначала выполните E0\'\n    with np.load(path) as z:\n        agreement = float((masked==z[\'pred_b5\']).mean())\n    assert agreement > .999, f\'Cached agreement={agreement}\'\n    selected = np.zeros((8,16),bool); selected[3:5,6:10] = True\n    routed = predict(model,state,img,{2:token_mask(selected,2)})\n    assert routed.shape == SIZE\n    write_json(RESULTS/\'smoke.json\', {\'complete\':True,\'all_true_equal\':True,\'cached_agreement\':agreement,\'selected_mask_forward_ok\':True})\n    print(\'Smoke OK. Совпадение с кэшем:\',agreement, flush=True)\n\n\ndef selftest():\n    from transformers import SegformerConfig, SegformerForSemanticSegmentation\n    from e8_cityscapes import install\n    torch.manual_seed(0)\n    cfg=SegformerConfig(num_channels=3,num_encoder_blocks=4,depths=[1,1,2,1],hidden_sizes=[8,16,32,64],num_attention_heads=[1,2,4,8],sr_ratios=[8,4,2,1],decoder_hidden_size=16,num_labels=19,drop_path_rate=0.)\n    m=SegformerForSemanticSegmentation(cfg).eval()\n    x=torch.randn(1,3,64,128)\n    with torch.inference_mode():\n        original=m(pixel_values=x).logits\n        states=install(m)\n        for st,s in states.items():\n            s.mask=torch.ones(1,(64//(4*2**st))*(128//(4*2**st)),1,dtype=torch.bool)\n        assert torch.equal(original,m(pixel_values=x).logits)\n        outputs=[]; handles=[]\n        for st,s in states.items():\n            s.mask=torch.zeros_like(s.mask)\n            for block in m.segformer.encoder.block[st]:\n                handles.append(block.register_forward_hook(lambda mod,inp,out:outputs.append(torch.equal(out[0],mod.state.entry))))\n        m(pixel_values=x)\n        for h in handles: h.remove()\n        assert outputs and all(outputs)\n    print(\'Tiny SegFormer: all-true / all-false OK\',flush=True)\n\nif __name__==\'__main__\':\n    p=argparse.ArgumentParser();p.add_argument(\'stage\',choices=[\'cache\',\'scores\',\'smoke\',\'e8\',\'selftest\']);args=p.parse_args()\n    if args.stage!=\'selftest\': assert torch.cuda.is_available(), \'Нужен GPU runtime\'\n    RESULTS.mkdir(parents=True,exist_ok=True);CACHE.mkdir(parents=True,exist_ok=True)\n    import transformers\n    write_json(RESULTS/\'environment.json\', {\'signature\':SIGNATURE,\'python\':platform.python_version(),\'torch\':torch.__version__,\'transformers\':transformers.__version__,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\'size\':SIZE,\'tile\':TILE,\'domains\':DOMAINS,\'training\':False})\n    if args.stage==\'e8\':\n        from e8_cityscapes import main\n        main()\n    else: globals()[args.stage]()\n'}
for name, text in SOURCES.items():
    path = EXP/name
    if path.exists():
        assert path.read_text() == text, f'Изменён код: {path}. Для новой версии поменяй RUN_NAME.'
    else: path.write_text(text)
code_copy = SAVE/'source'
code_copy.mkdir(exist_ok=True)
for name in SOURCES: shutil.copy2(EXP/name, code_copy/name)
manifest = {'sha256':{k:hashlib.sha256(v.encode()).hexdigest() for k,v in SOURCES.items()},
            'domains':['cs_fit','cs_test'], 'size':[512,1024], 'tile':64,
            'budgets':[0.1,0.2], 'frozen_models':True, 'rank_transductive':True}
manifest_path = SAVE/'run_config.json'
if manifest_path.exists():
    assert json.loads(manifest_path.read_text()) == manifest, 'Конфигурация изменилась: выбери новый RUN_NAME'
else: manifest_path.write_text(json.dumps(manifest, indent=2))
print('Код записан:', EXP)


In [ ]:
# 5. Восстановление после сброса VM и запускатель с логами.
import tarfile, time
for domain in ['cs_fit','cs_test']:
    backup = SAVE/'backups'/f'cache_{domain}.tar'
    if backup.is_file() and not (CACHE/domain).exists():
        print('Восстанавливаем:', backup, flush=True)
        with tarfile.open(backup) as tar:
            members = tar.getmembers()
            for m in members:
                assert m.isfile() or m.isdir()
                assert (CACHE/m.name).resolve().is_relative_to(CACHE.resolve())
            tar.extractall(CACHE)
for name in ['e8_knn_bank.npy','e8_tiles_cityscapes.npz']:
    src = SAVE/name
    if src.is_file() and not (CACHE/name).exists(): shutil.copy2(src,CACHE/name)

def run(stage):
    log_path = SAVE/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск:', stage, '| Лог:', log_path, flush=True)
    started = time.monotonic()
    with log_path.open('w') as log:
        process = subprocess.Popen([sys.executable,'-u','runner.py',stage],cwd=EXP,
                                   env=os.environ.copy(), stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30)
                    break
                except subprocess.TimeoutExpired:
                    tail = log_path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{stage}: {(time.monotonic()-started)/60:.1f} мин | '+ ' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait()
            raise
    print(log_path.read_text(errors='replace')[-8000:])
    assert status == 0, f'Ошибка {status}: смотри лог {log_path}'
    print(f'Завершено за {(time.monotonic()-started)/60:.1f} мин')


## Новая часть E9

**Архитектура:** `Conv1×1(160→128) → GELU → Conv1×1(128→640)`, 103 168 параметров. Первые 320 каналов предсказывают `mid`, следующие 320 — `final`. Сетка 32×64, как у stage 3 B5; пространственное выравнивание не требует интерполяции.

**Данные:** teacher-цели получаем замороженной B5. Вход — F3 замороженной B0. Нормализация и оптимизация только на обучающей части Frankfurt. Наименьшая по числу кадров последовательность Frankfurt целиком выделяется для validation/early stopping; точные списки сохраняются. Проверка — Lindau+Münster (`cs_test`).

**Цель:** среднее MSE двух состояний после нормализации каждого канала по обучающей части. Это дистилляция представлений, а не сегментационная cross-entropy; GT-маски в эту потерю не входят, включая регионы ignore. AdamW, lr=0.001, batch=2, максимум 20 эпох, patience=4. Лучшую эпоху выбираем по validation MSE, не по результату cs_test.

Все эти параметры фиксированы до запуска. Это первый вариант адаптера, не подбор архитектуры по cs_test.


In [ ]:
# 6. Проверка готовых кэшей и новая папка E9.
for domain,n in [('cs_fit',267),('cs_test',233)]:
    paths=list((CACHE/domain).glob('*.npz'))
    assert len(paths)==n, f'Неполный кэш {domain}: {len(paths)}/{n}. Восстанови E8 из backups.'
for p in [CACHE/'e8_tiles_cityscapes.npz',RESULTS/'e0_cityscapes.json']:
    assert p.is_file(), f'Не найден {p}; восстанови завершённый запуск E8.'
E9_OUT=SAVE/'e9_adapter_v1'
E9_LOCAL=WORK/'e9_adapter_v1'
E9_OUT.mkdir(exist_ok=True);E9_LOCAL.mkdir(exist_ok=True)
os.environ.update(E9_OUT=str(E9_OUT),E9_LOCAL=str(E9_LOCAL))
print('E9:',E9_OUT)


In [ ]:
# 7. Код E9 встроен в notebook; репозиторий и старые результаты не меняются.
E9_SOURCE='"""E9 diagnostic: B0 F3 -> B5 mid/final state. Frozen segmenters, teacher-assisted routing."""\nimport os\nos.environ[\'USE_TF\']=\'0\'\nimport argparse, csv, hashlib, json, time, shutil\nfrom pathlib import Path\nfrom types import SimpleNamespace\nfrom collections import Counter\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import Dataset, DataLoader\nimport torch.nn.functional as F\nfrom common import CACHE, RESULTS, SIZE, TILE, SegSet, load_model, confusion, miou, top_tiles\n\nOUT=Path(os.environ[\'E9_OUT\'])\nLOCAL=Path(os.environ[\'E9_LOCAL\'])\nCONFIG={\'schema\':\'e9-mid-final-adapter-v1\',\'seed\':0,\'input\':\'B0 normalized encoder F3 (stage index 2 output)\',\n        \'input_channels\':160,\'target_channels\':320,\'hidden\':128,\'target_states\':[\'mid\',\'final\'],\n        \'epochs\':20,\'patience\':4,\'lr\':.001,\'weight_decay\':.0001,\'batch\':2,\n        \'budgets\':[.1,.2],\'evaluation_domains\':[\'cs_test\'],\'methods\':[\'rank\'],\n        \'teacher_assisted\':True,\'keep_real_b5_early_stages\':True,\'routed_entry\':\'real B5\',\n        \'loss\':\'mean channel-standardized MSE, mid and final equal weight\',\n        \'fit_domains\':[\'cs_fit\'],\'size\':[512,1024], \'compute_benchmark\':False}\n\n\ndef write_json(path,data):\n    path.parent.mkdir(parents=True,exist_ok=True)\n    p=path.with_suffix(path.suffix+\'.part\');p.write_text(json.dumps(data,indent=2));p.replace(path)\n\ndef atomic_torch(path,data):\n    p=path.with_suffix(\'.pt.part\');torch.save(data,p);p.replace(path)\n\ndef sha(path):return hashlib.sha256(path.read_bytes()).hexdigest()\n\ndef initialize():\n    OUT.mkdir(parents=True,exist_ok=True);LOCAL.mkdir(parents=True,exist_ok=True)\n    cfg=dict(CONFIG,source_sha256=sha(Path(__file__)),\n             common_sha256=sha(Path(__file__).parent/\'common.py\'),\n             tile_features_sha256=sha(CACHE/\'e8_tiles_cityscapes.npz\'))\n    p=OUT/\'config.json\'\n    if p.exists():assert json.loads(p.read_text())==cfg,\'Configuration differs: use a new E9_OUT\'\n    else:write_json(p,cfg)\n    return cfg\n\n\nclass ContextBlock(nn.Module):\n    def __init__(self,layer,state,index,n):\n        super().__init__();self.layer=layer;self.state=state;self.index=index;self.n=n\n    def forward(self,x,*args,**kwargs):\n        s=self.state\n        if s.recording:\n            if self.index==0:s.real[\'entry\']=x\n            if self.index==self.n//2:s.real[\'mid\']=x\n            out=self.layer(x,*args,**kwargs)\n            if self.index==self.n-1:s.real[\'final\']=out[0]\n            return out\n        if s.mask is None:return self.layer(x,*args,**kwargs)\n        ctx=s.context\n        out=self.layer(torch.where(s.mask,x,ctx),*args,**kwargs)\n        y=torch.where(s.mask,out[0],ctx)\n        if self.index==self.n-1:y=torch.where(s.mask,y,s.output)\n        return (y,)+tuple(out[1:])\n\n\ndef install(model):\n    state=SimpleNamespace(recording=False,real={},mask=None,context=None,output=None)\n    layers=model.segformer.encoder.block[2]\n    for i,layer in enumerate(layers):layers[i]=ContextBlock(layer,state,i,len(layers))\n    return state\n\n@torch.inference_mode()\ndef forward(model,state,img,recording=False,mask=None,context=None,output=None):\n    state.recording=recording;state.mask=mask;state.context=context;state.output=output\n    try:return model(pixel_values=img).logits\n    finally:state.recording=False;state.mask=None;state.context=None;state.output=None\n\ndef prediction(logits):\n    return F.interpolate(logits.float(),SIZE,mode=\'bilinear\',align_corners=False).argmax(1)[0].byte().cpu().numpy()\n\ndef target_path(domain,name):return LOCAL/\'targets\'/domain/f\'{name}.npz\'\n\ndef check_target(path):\n    if not path.is_file():return False\n    with np.load(path,allow_pickle=False) as z:\n        assert z[\'x\'].shape==(160,32,64)\n        assert all(z[k].shape==(320,32,64) for k in [\'mid\',\'final\'])\n        assert all(np.isfinite(z[k]).all() for k in [\'x\',\'mid\',\'final\'])\n    return True\n\n\ndef split_ids(names):\n    # City+sequence, not neighboring-frame random split.\n    groups={n:\'_\'.join(n.split(\'_\')[:2]) for n in names}\n    counts=Counter(groups.values())\n    assert len(counts)>=2,\'Need at least two Frankfurt sequences\'\n    val_group=min(counts,key=lambda g:(counts[g],g))\n    train=[n for n in names if groups[n]!=val_group]\n    val=[n for n in names if groups[n]==val_group]\n    assert train and val and len(val)<len(train),(counts, val_group)\n    return {\'train\':train,\'val\':val,\'val_sequence\':val_group,\'sequence_counts\':dict(counts),\n            \'note\':\'Validation is smallest Frankfurt sequence; cs_test never used for selection.\'}\n\n@torch.inference_mode()\ndef export_targets():\n    b0=load_model(\'b0\').requires_grad_(False)\n    b5=load_model(\'b5\').requires_grad_(False);state=install(b5)\n    assert len(b5.segformer.encoder.block[2])==40,\'Expected B5 stage 3 depth=40\'\n    for domain in [\'cs_fit\',\'cs_test\']:\n        data=SegSet(domain)\n        expected=267 if domain==\'cs_fit\' else 233\n        assert len(data)==expected\n        assert [p.stem for p,_ in data.items]==[p.stem for p in sorted((CACHE/domain).glob(\'*.npz\'))]\n        start=time.monotonic()\n        for i,(image,gt_path) in enumerate(data.items):\n            name=image.stem;p=target_path(domain,name)\n            backup=OUT/\'targets\'/domain/p.name\n            p.parent.mkdir(parents=True,exist_ok=True);backup.parent.mkdir(parents=True,exist_ok=True)\n            if not p.exists() and backup.exists():shutil.copy2(backup,p)\n            if not check_target(p):\n                img,_,_=data[i];x=img[None].cuda().half()\n                f3=b0(pixel_values=x,output_hidden_states=True).hidden_states[2][0]\n                pred=prediction(forward(b5,state,x,recording=True))\n                with np.load(CACHE/domain/p.name) as z:agree=float((pred==z[\'pred_b5\']).mean())\n                assert agree>.999,f\'B5/cache mismatch {name}: {agree}\'\n                packed={\'x\':f3.cpu().numpy()}\n                for k in [\'mid\',\'final\']:\n                    v=state.real[k]\n                    assert v.shape==(1,2048,320),v.shape\n                    packed[k]=v[0].T.reshape(320,32,64).cpu().numpy()\n                tmp=p.with_suffix(\'.npz.part\')\n                with tmp.open(\'wb\') as f:np.savez_compressed(f,**packed)\n                tmp.replace(p);state.real={}\n            if not backup.exists():\n                tmp=backup.with_suffix(\'.npz.part\');shutil.copy2(p,tmp);tmp.replace(backup)\n            if (i+1)%10==0 or i+1==len(data):\n                print(f\'Targets {domain}: {i+1}/{len(data)}, {(time.monotonic()-start)/60:.1f} мин\',flush=True)\n    names=[p.stem for p,_ in SegSet(\'cs_fit\').items]\n    splits=split_ids(names)\n    p=OUT/\'splits.json\'\n    if p.exists():assert json.loads(p.read_text())==splits\n    else:write_json(p,splits)\n    print(\'Train:\',len(splits[\'train\']),\'val:\',len(splits[\'val\']),\'sequence:\',splits[\'val_sequence\'],flush=True)\n    write_json(OUT/\'targets_complete.json\',{\'complete\':True,\'domains\':{\'cs_fit\':267,\'cs_test\':233}})\n\n\ndef ensure_local(domain,names):\n    for n in names:\n        p=target_path(domain,n)\n        if not p.exists():\n            p.parent.mkdir(parents=True,exist_ok=True)\n            shutil.copy2(OUT/\'targets\'/domain/p.name,p)\n        check_target(p)\n\n\ndef normalization(names):\n    p=OUT/\'normalization.npz\'\n    if p.exists():\n        with np.load(p) as z:return {k:z[k] for k in z.files}\n    sums={k:np.zeros(d,np.float64) for k,d in [(\'x\',160),(\'mid\',320),(\'final\',320)]}\n    squares={k:v.copy() for k,v in sums.items()};n=0\n    for name in names:\n        with np.load(target_path(\'cs_fit\',name)) as z:\n            for k in sums:\n                a=z[k].astype(np.float64).reshape(len(sums[k]),-1)\n                sums[k]+=a.sum(1);squares[k]+=(a*a).sum(1)\n            n+=a.shape[1]\n    result={}\n    for k in sums:\n        mean=sums[k]/n\n        std=np.sqrt(np.maximum(squares[k]/n-mean**2,1e-8))\n        result[k+\'_mean\']=mean.astype(np.float32)\n        result[k+\'_std\']=std.astype(np.float32)\n    np.savez(p,**result)\n    return result\n\nclass Adapter(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.net=nn.Sequential(nn.Conv2d(160,128,1),nn.GELU(),nn.Conv2d(128,640,1))\n    def forward(self,x):return self.net(x)\n\nclass Maps(Dataset):\n    def __init__(self,names,norm):self.names=names;self.norm=norm\n    def __len__(self):return len(self.names)\n    def __getitem__(self,i):\n        with np.load(target_path(\'cs_fit\',self.names[i])) as z:\n            def standard(k):return (z[k].astype(np.float32)-self.norm[k+\'_mean\'][:,None,None])/self.norm[k+\'_std\'][:,None,None]\n            return torch.from_numpy(standard(\'x\')),torch.from_numpy(np.concatenate([standard(\'mid\'),standard(\'final\')]))\n\n@torch.no_grad()\ndef val_loss(model,loader,device):\n    model.eval();s=0.;n=0\n    for x,y in loader:\n        x=x.to(device);y=y.to(device)\n        s+=F.mse_loss(model(x),y,reduction=\'sum\').item();n+=y.numel()\n    return s/n\n\n\ndef train(device="cuda"):\n    torch.manual_seed(0);np.random.seed(0)\n    splits=json.loads((OUT/\'splits.json\').read_text())\n    ensure_local(\'cs_fit\',splits[\'train\']+splits[\'val\'])\n    norm=normalization(splits[\'train\'])\n    done=OUT/\'training_complete.json\'\n    if done.exists():\n        d=json.loads(done.read_text())\n        assert d[\'checkpoint_sha256\']==sha(OUT/\'adapter_best.pt\')\n        print(\'Адаптер уже обучен, пропускаем.\',flush=True);return\n    model=Adapter().to(device)\n    optimizer=torch.optim.AdamW(model.parameters(),lr=CONFIG[\'lr\'],weight_decay=CONFIG[\'weight_decay\'])\n    val_loader=DataLoader(Maps(splits[\'val\'],norm),batch_size=2,num_workers=0)\n    last=OUT/\'adapter_last.pt\';best=float(\'inf\');bad=0;history=[];start=0;best_epoch=None\n    if last.exists():\n        checkpoint=torch.load(last,map_location=device,weights_only=True)\n        model.load_state_dict(checkpoint[\'model\']);optimizer.load_state_dict(checkpoint[\'optimizer\'])\n        best=checkpoint[\'best\'];bad=checkpoint[\'bad\'];history=checkpoint[\'history\'];start=checkpoint[\'epoch\']\n        best_epoch=checkpoint[\'best_epoch\']\n        print(\'Возобновляем после эпохи\',start,flush=True)\n    for epoch in range(start,CONFIG[\'epochs\']):\n        if bad>=CONFIG[\'patience\']:break\n        generator=torch.Generator().manual_seed(CONFIG[\'seed\']+epoch)\n        loader=DataLoader(Maps(splits[\'train\'],norm),batch_size=CONFIG[\'batch\'],shuffle=True,generator=generator,num_workers=0)\n        model.train();total=0.;count=0\n        for x,y in loader:\n            x=x.to(device);y=y.to(device);optimizer.zero_grad(set_to_none=True)\n            pred=model(x);loss=F.mse_loss(pred,y)\n            assert torch.isfinite(loss),\'Nonfinite loss\'\n            loss.backward();torch.nn.utils.clip_grad_norm_(model.parameters(),5.)\n            optimizer.step();total+=loss.item()*y.numel();count+=y.numel()\n        val=val_loss(model,val_loader,device)\n        row={\'epoch\':epoch+1,\'train_normalized_mse\':total/count,\'val_normalized_mse\':val}\n        history.append(row)\n        if val<best:\n            best=val;bad=0;best_epoch=epoch+1\n            atomic_torch(OUT/\'adapter_best.pt\',{\'model\':model.state_dict(),\'epoch\':best_epoch,\'val_mse\':best})\n        else:bad+=1\n        atomic_torch(last,{\'model\':model.state_dict(),\'optimizer\':optimizer.state_dict(),\'epoch\':epoch+1,\n                          \'best\':best,\'bad\':bad,\'best_epoch\':best_epoch,\'history\':history})\n        write_json(OUT/\'training_history.json\',history)\n        print(row,flush=True)\n    best_checkpoint=torch.load(OUT/\'adapter_best.pt\',map_location=device,weights_only=True)\n    model.load_state_dict(best_checkpoint[\'model\']);model.eval()\n    diag={k:{\'error\':0.,\'energy\':0.,\'mean_error\':0.} for k in [\'mid\',\'final\']}\n    with torch.no_grad():\n        for name in splits[\'val\']:\n            with np.load(target_path(\'cs_fit\',name)) as z:\n                x=(z[\'x\'].astype(np.float32)-norm[\'x_mean\'][:,None,None])/norm[\'x_std\'][:,None,None]\n                y=model(torch.from_numpy(x)[None].to(device))[0].cpu().numpy()\n                for i,k in enumerate([\'mid\',\'final\']):\n                    target=z[k].astype(np.float64)\n                    output=y[i*320:(i+1)*320]*norm[k+\'_std\'][:,None,None]+norm[k+\'_mean\'][:,None,None]\n                    diag[k][\'error\']+=float(((output-target)**2).sum())\n                    diag[k][\'energy\']+=float((target**2).sum())\n                    diag[k][\'mean_error\']+=float(((norm[k+\'_mean\'][:,None,None]-target)**2).sum())\n    diagnostics={k:{\'relative_squared_error\':d[\'error\']/max(d[\'energy\'],1e-12),\n                    \'error_vs_train_mean_baseline\':d[\'error\']/max(d[\'mean_error\'],1e-12)} for k,d in diag.items()}\n    write_json(OUT/\'adapter_validation.json\',diagnostics)\n    summary={\'complete\':True,\'best_epoch\':best_epoch,\'val_normalized_mse\':best,\n             \'parameters\':sum(p.numel() for p in model.parameters()),\'train_frames\':len(splits[\'train\']),\n             \'val_frames\':len(splits[\'val\']),\'checkpoint_sha256\':sha(OUT/\'adapter_best.pt\'),\n             \'normalization_sha256\':sha(OUT/\'normalization.npz\')}\n    write_json(done,summary);print(summary,diagnostics,flush=True)\n\n@torch.inference_mode()\ndef estimate(model,norm,array,device):\n    x=(array.astype(np.float32)-norm[\'x_mean\'][:,None,None])/norm[\'x_std\'][:,None,None]\n    y=model(torch.from_numpy(x)[None].to(device))[0]\n    result={}\n    for i,k in enumerate([\'mid\',\'final\']):\n        output=y[i*320:(i+1)*320]\n        output=output*torch.as_tensor(norm[k+\'_std\'],device=device)[:,None,None]+torch.as_tensor(norm[k+\'_mean\'],device=device)[:,None,None]\n        result[k]=output.flatten(1).T[None].to(dtype=torch.float16 if str(device).startswith(\'cuda\') else torch.float32)\n    return result\n\n\ndef rank(x):return x.ravel().argsort().argsort().reshape(x.shape)\n\n@torch.inference_mode()\ndef evaluate(device="cuda"):\n    summary=json.loads((OUT/\'training_complete.json\').read_text())\n    assert summary[\'checkpoint_sha256\']==sha(OUT/\'adapter_best.pt\')\n    assert summary[\'normalization_sha256\']==sha(OUT/\'normalization.npz\')\n    adapter=Adapter().to(device).eval().requires_grad_(False)\n    adapter.load_state_dict(torch.load(OUT/\'adapter_best.pt\',map_location=device,weights_only=True)[\'model\'])\n    with np.load(OUT/\'normalization.npz\') as z:norm={k:z[k] for k in z.files}\n    model=load_model(\'b5\',device=device).requires_grad_(False);state=install(model)\n    feats=np.load(CACHE/\'e8_tiles_cityscapes.npz\')\n    e8b_path=RESULTS.parent/\'e8b_v1/e8b_cityscapes.json\'\n    old=json.loads(e8b_path.read_text()) if e8b_path.exists() else {}\n    modes=[\'true_context_true_output\',\'pred_context_true_output\',\'true_context_pred_output\',\'pred_context_pred_output\',\'mean_context_mean_output\']\n    report_path=OUT/\'e9_routing.json\'\n    report=json.loads(report_path.read_text()) if report_path.exists() else {}\n    for domain in CONFIG[\'evaluation_domains\']:\n        if domain in report:\n            print(domain,\'готов, пропускаем\',flush=True);continue\n        data=SegSet(domain);names=[p.stem for p,_ in data.items]\n        ensure_local(domain,names)\n        assert names==[p.stem for p in sorted((CACHE/domain).glob(\'*.npz\'))]\n        score=rank(feats[f\'{domain}__entropy__mean\'])+rank(feats[f\'{domain}__knn__mean\'])\n        masks=[top_tiles(score,b) for b in CONFIG[\'budgets\']]\n        n=len(data);c=len(modes)+2\n        confs=np.zeros((n,len(masks),c,19,19),np.int64)\n        agreements=np.zeros((len(masks),len(modes)+1,2),np.int64)\n        start=time.monotonic()\n        dtype=torch.float16 if str(device).startswith(\'cuda\') else torch.float32\n        mean={k:torch.as_tensor(norm[k+\'_mean\'],device=device)[None,None,:].expand(1,2048,320).to(dtype=dtype) for k in [\'mid\',\'final\']}\n        for i,name in enumerate(names):\n            img,_,_=data[i];img=img[None].to(device=device,dtype=dtype)\n            full_pred=prediction(forward(model,state,img,recording=True))\n            with np.load(CACHE/domain/f\'{name}.npz\') as z:gt=z[\'gt\'];b0=z[\'pred_b0\'];b5=z[\'pred_b5\']\n            assert (full_pred==b5).mean()>.999,f\'B5/cache mismatch: {name}\'\n            with np.load(target_path(domain,name)) as z:pred=estimate(adapter,norm,z[\'x\'],device)\n            true=state.real\n            options=[(true[\'mid\'],true[\'final\']),(pred[\'mid\'],true[\'final\']),\n                     (true[\'mid\'],pred[\'final\']),(pred[\'mid\'],pred[\'final\']),(mean[\'mid\'],mean[\'final\'])]\n            # All-true with predicted contexts/output must still reproduce the normal pass.\n            if i==0:\n                ones=torch.ones((1,2048,1),device=device,dtype=torch.bool)\n                check=prediction(forward(model,state,img,mask=ones,context=pred[\'mid\'],output=pred[\'final\']))\n                assert np.array_equal(check,full_pred),\'All-true prediction mismatch\'\n            for j,m in enumerate(masks):\n                px=m[i].repeat(TILE,0).repeat(TILE,1)\n                tm=torch.from_numpy(m[i]).to(device).repeat_interleave(4,0).repeat_interleave(4,1).reshape(1,2048,1)\n                confs[i,j,0]=confusion(b0,gt)\n                confs[i,j,1]=confusion(np.where(px,b5,b0),gt)\n                agreements[j,0]+=[int((b0==b5)[px].sum()),int(px.sum())]\n                for k,(context,output) in enumerate(options):\n                    modified=prediction(forward(model,state,img,mask=tm,context=context,output=output))\n                    confs[i,j,k+2]=confusion(np.where(px,modified,b0),gt)\n                    agreements[j,k+1]+=[int((modified==b5)[px].sum()),int(px.sum())]\n            state.real={}\n            if (i+1)%5==0 or i+1==n:\n                elapsed=time.monotonic()-start\n                print(f\'E9 {domain}: {i+1}/{n}, {elapsed/60:.1f} мин, ETA ~{elapsed/(i+1)*(n-i-1)/60:.1f} мин\',flush=True)\n        rows=[]\n        for j,budget in enumerate(CONFIG[\'budgets\']):\n            values=[float(miou(v)) for v in confs[:,j].sum(0)]\n            for k,mode in enumerate(modes):\n                a=agreements[j,k+1]\n                row={\'method\':\'rank\',\'budget\':budget,\'variant\':mode,\'miou\':values[k+2],\n                     \'miou_b0\':values[0],\'miou_full_tiles\':values[1],\n                     \'delta_vs_b0_pp\':values[k+2]-values[0],\n                     \'delta_vs_true_context_output_pp\':values[k+2]-values[2],\n                     \'delta_vs_full_tiles_pp\':values[k+2]-values[1],\n                     \'agree_b5_percent\':100*int(a[0])/max(int(a[1]),1)}\n                row[\'agree_b0_percent\']=100*int(agreements[j,0,0])/max(int(agreements[j,0,1]),1)\n                if k==0 and domain in old and f\'rank@{budget}\' in old[domain]:\n                    previous=old[domain][f\'rank@{budget}\']\n                    delta=row[\'miou\']-previous[\'miou_mid\']\n                    row[\'control_delta_vs_e8b_mid_pp\']=delta\n                    assert abs(delta)<.01,f\'True-control differs from E8b mid: {delta}\'\n                rows.append(row);print(row,flush=True)\n        np.savez_compressed(OUT/f\'{domain}_frame_confusions.npz\',confs=confs,\n                            names=np.array(names),budgets=np.array(CONFIG[\'budgets\']),\n                            modes=np.array([\'b0\',\'full_tiles\']+modes),agreements=agreements)\n        report[domain]=rows;write_json(report_path,report)\n    flat=[dict(domain=d,**r) for d,rows in report.items() for r in rows]\n    fields=list(dict.fromkeys(k for r in flat for k in r))\n    with (OUT/\'e9_routing.csv\').open(\'w\',newline=\'\') as f:\n        writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(flat)\n\n\ndef bootstrap():\n    # Paired dataset mIoU, recomputed from summed confusion matrices.\n    rows=[];rng=np.random.default_rng(0)\n    for domain in CONFIG[\'evaluation_domains\']:\n        with np.load(OUT/f\'{domain}_frame_confusions.npz\') as z:\n            conf=z[\'confs\'];names=z[\'names\'];modes=list(z[\'modes\']);budgets=z[\'budgets\']\n        for unit in [\'frame\',\'sequence\']:\n            groups=np.arange(len(names)).astype(str) if unit==\'frame\' else np.array([\'_\'.join(n.split(\'_\')[:2]) for n in names])\n            unique=sorted(set(groups));grouped=np.stack([conf[groups==g].sum(0) for g in unique])\n            for j,budget in enumerate(budgets):\n                for reference in [\'true_context_true_output\',\'mean_context_mean_output\',\'full_tiles\']:\n                    a=modes.index(\'pred_context_pred_output\');b=modes.index(reference)\n                    diffs=[]\n                    for _ in range(1000):\n                        idx=rng.integers(0,len(unique),len(unique));c=grouped[idx,j].sum(0)\n                        diffs.append(miou(c[a])-miou(c[b]))\n                    total=grouped[:,j].sum(0)\n                    ci=np.percentile(diffs,[2.5,97.5])\n                    rows.append({\'domain\':domain,\'budget\':float(budget),\'unit\':unit,\'groups\':len(unique),\n                                 \'comparison\':f\'pred_both minus {reference}\',\n                                 \'delta_miou_pp\':float(miou(total[a])-miou(total[b])),\n                                 \'ci_low\':float(ci[0]),\'ci_high\':float(ci[1])})\n    write_json(OUT/\'bootstrap.json\',rows)\n    for row in rows:print(row,flush=True)\n\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'stage\',choices=[\'export_targets\',\'train\',\'evaluate\',\'bootstrap\'])\n    args=parser.parse_args();initialize()\n    if args.stage!=\'bootstrap\':assert torch.cuda.is_available(),\'Need GPU\'\n    import transformers,platform\n    write_json(OUT/\'environment.json\',{\'python\':platform.python_version(),\'torch\':torch.__version__,\n        \'transformers\':transformers.__version__,\'gpu\':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None})\n    globals()[args.stage]()\nif __name__==\'__main__\':main()\n'
e9_script=EXP/'e9_adapter.py'
if e9_script.exists():
    assert e9_script.read_text()==E9_SOURCE, 'Версия скрипта изменилась: для новой версии используй отдельную папку E9.'
else:e9_script.write_text(E9_SOURCE)
saved_source=E9_OUT/'e9_adapter.py'
if saved_source.exists():
    assert saved_source.read_text()==E9_SOURCE, 'В папке E9 другая версия скрипта: выбери новую E9_OUT.'
else:saved_source.write_text(E9_SOURCE)
print('Код готов')


In [ ]:
# 8. Запускатель с прогрессом и сохранением лога на Drive.
def run_e9(stage):
    log_path=E9_OUT/f'{stage}_{time.strftime("%Y%m%d_%H%M%S")}.log'
    print('Запуск:',stage,'| Лог:',log_path,flush=True)
    start=time.monotonic()
    with log_path.open('w') as log:
        process=subprocess.Popen([sys.executable,'-u',str(e9_script),stage],cwd=EXP,
                                 env=os.environ.copy(),stdout=log,stderr=subprocess.STDOUT)
        try:
            while True:
                try:
                    status=process.wait(timeout=30);break
                except subprocess.TimeoutExpired:
                    tail=log_path.read_text(errors='replace').splitlines()[-2:]
                    print(f'{stage}: {(time.monotonic()-start)/60:.1f} мин | '+' | '.join(tail),flush=True)
        except KeyboardInterrupt:
            process.terminate();process.wait();raise
    print(log_path.read_text(errors='replace')[-14000:])
    assert status==0, f'Ошибка {status}. Лог: {log_path}'
    print('Завершено за',round((time.monotonic()-start)/60,1),'мин')


In [ ]:
# 9. Новые teacher-карты: B0 F3, B5 mid и final.
# B0/B5 в eval + inference_mode, requires_grad=False. Готовые записи пропускаются.
# Каждая запись резервируется на Drive; при сбросе VM уже готовые цели восстанавливаются.
run_e9('export_targets')


In [ ]:
# 10. Обучение ТОЛЬКО адаптера. Сохраняются best и last checkpoint, optimizer и история.
# После сброса среды обучение продолжается с последней завершённой эпохи.
run_e9('train')
print('Диагностика на отдельной последовательности Frankfurt:')
print((E9_OUT/'adapter_validation.json').read_text())
print('error_vs_train_mean_baseline < 1 означает: адаптер восстанавливает состояние лучше постоянного среднего из train.')


## Проверка качества: четыре замены + простой контроль

Для обоих бюджетов 10%/20% используется тот же `rank` (entropy+kNN) и те же маски тайлов, что в E8b. Оракул в первый E9 не включён, чтобы сократить прогон и сосредоточиться на применимом способе выбора.

| Вариант | Контекст во всех блоках stage 3 | Выход вне выбранных тайлов |
|---|---|---|
| `true_context_true_output` | настоящая середина B5 | настоящий final B5 |
| `pred_context_true_output` | предсказанный mid | настоящий final B5 |
| `true_context_pred_output` | настоящий mid B5 | предсказанный final |
| `pred_context_pred_output` | предсказанный mid | предсказанный final |
| `mean_context_mean_output` | постоянный средний mid из train | постоянный средний final из train |

Первый вариант должен повторить E8b `mid`. Второй и третий локализуют потерю при замене каждой подсказки. Четвёртый — основной результат адаптера. Пятый проверяет, полезно ли индивидуальное предсказание, а не только среднее распределение.

**У всех вариантов выбранные токены начинают с настоящего входа B5.** Ранние карты B5 для декодера также настоящие. Вход B0 при оценке берётся из сохранённого кэша. Поэтому время выполнения здесь не является сквозной латентностью каскада.


In [ ]:
# 11. Сегментационная проверка только на cs_test. 11 проходов B5 на кадр.
run_e9('evaluate')


In [ ]:
# 12. Парный bootstrap mIoU: по кадрам и последовательностям.
# Эти города уже использовались в предыдущих экспериментах — проверка исследовательская.
run_e9('bootstrap')


In [ ]:
# 13. Итоги и архив. Teacher-карты и adapter_last.pt в архив не включаются.
import pandas as pd
from IPython.display import display
import zipfile
print('Обучение:')
print((E9_OUT/'training_complete.json').read_text())
display(pd.DataFrame(json.loads((E9_OUT/'training_history.json').read_text())))
print('Результаты cs_test:')
df=pd.read_csv(E9_OUT/'e9_routing.csv')
display(df[['domain','budget','variant','miou','delta_vs_b0_pp',
            'delta_vs_true_context_output_pp','delta_vs_full_tiles_pp','agree_b5_percent']])
display(pd.DataFrame(json.loads((E9_OUT/'bootstrap.json').read_text())))
archive=E9_OUT/'e9_adapter_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for p in E9_OUT.iterdir():
        if p.is_file() and p.name not in {archive.name,'adapter_last.pt'} and not p.name.endswith('.part'):
            z.write(p,p.name)
    for p in [RESULTS/'e0_cityscapes.json',RESULTS/'e8_cityscapes.json',
              SAVE/'e8b_v1/e8b_cityscapes.json',SAVE/'run_config.json',EXP/'common.py']:
        if p.is_file():z.write(p,'reference/'+p.name)
print('Пришли:',archive)
# Скачать:
# from google.colab import files
# files.download(str(archive))


## Что будет основанием для следующего шага

Сначала проверьте, что `true_context_true_output` воспроизводит E8b `mid`. Скрипт сверяет mIoU, если прошлый E8b JSON доступен.

Затем сравните `pred_context_pred_output` с этим эталоном, с постоянным средним контролем и с обычной подстановкой `full_tiles`. Высокое совпадение предсказаний само по себе недостаточно: важен mIoU и его потеря при замене подсказок.

Успех покажет, что конкретный адаптер способен заменить два состояния в teacher-assisted схеме на Cityscapes. Чтобы получить реальный каскад, затем потребуется заменить настоящий вход выбранных токенов и ранние карты B5, реализовать разреженное вычисление и замерить полное время. ACDC — отдельная проверка переноса с зафиксированными весами и настройками.

После сброса VM снова выполните подготовку и этапы E9 сверху вниз: готовые teacher-файлы восстановятся, обучение продолжится с завершённой эпохи, готовая оценка cs_test пропустится. Неполный оценочный домен считается заново.
